# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIRˆ2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is published under a Croissant schema at:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset Croissant metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review the record sets, fields, and their `@id`s.
We use the Croissant API to list all available record sets and fields, referencing by their `@id`s.

In [ ]:
# List all record sets and their fields by @id.

record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f'RecordSet name: {rs.name}\n  @id: {rs.id}')
    print("  Fields:")
    for field in rs.fields:
        column_ids = [c.id for c in field.columns]
        print(f"    - Field: {field.name:30} @id: {field.id}  (columns: {column_ids})")
    print()

## 3. Data Extraction

Load data from the desired record set(s) into DataFrames for analysis. 

Below, update the `record_set_ids` list with the targets you want to load, as obtained from the previous overview.

> **Tip:** If you are unsure which record set to begin with, the overview printout above will show their `@id`s (e.g., `cr:main_table` or similar.)


In [ ]:
# For demonstration, find a record set with tabular data.
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading data from RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {df.shape[0]} records with columns: {df.columns.tolist()}")
    else:
        print("  No records found in this RecordSet.")
print("\nAvailable DataFrames:", list(dataframes.keys()))

# Inspect the first (largest) DataFrame
if dataframes:
    main_record_set_id = max(dataframes, key=lambda k: dataframes[k].shape[0])
    print(f"\nPreview of main table ({main_record_set_id}):")
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Below are typical steps such as filtering, normalization, and grouping for protein analysis.

You can adapt the field and group keys to those printed in the extraction step above (e.g. `Abundance`, `MW`, or other numeric fields referenced by their `@id`).

In [ ]:
# Try to select a common numeric field (adjust to match the names/id's in your dataset)

record_set_id = main_record_set_id if dataframes else None
df = dataframes[record_set_id] if record_set_id is not None else pd.DataFrame()

# Guess a numeric field
candidate_fields = [c for c in df.columns if any(substr in c.lower() for substr in ["abund", "mw", "count", "coverage", "length", "score", "peptide"])]
numeric_field_id = candidate_fields[0] if candidate_fields else (df.columns[0] if not df.empty else None)
print(f"Using numeric field: {numeric_field_id}")

if numeric_field_id is not None:
    # Remove non-numeric values safely
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isna().all() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold:.2f} (mean)")

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to group by a likely categorical field
    group_candidates = [c for c in df.columns if c != numeric_field_id and df[c].dtype == 'object']
    group_field_id = group_candidates[0] if group_candidates else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print('No numeric field found for analysis.')

## 5. Visualization

Visualize the distribution of the chosen numeric field, and (if grouping field exists) the grouped means.

> Note: Visualization requires matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id is not None and not df[numeric_field_id].isna().all():
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Grouped bar plot
    if 'grouped_df' in locals() and group_field_id:
        grouped_df.plot.bar(x=group_field_id, y=numeric_field_id, figsize=(10,3), legend=False)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion

This notebook demonstrated how to:
- Load FAIRˆ2-compliant protein experiment data using the Croissant standard and the `mlcroissant` library,
- Explore data record sets, fields, and column ids using their `@id`s,
- Load one or more data tables to pandas dataframes,
- Perform simple EDA including filtering and normalization,
- Visualize numeric distributions and group statistics.

Next steps may include specific statistical tests, feature engineering, or applying machine learning approaches to the protein dataset.

---
_Notebook generated via `mlcroissant` example structure for the FAIRˆ2 dataset._